# RAG Pipeline — Embed Data to Redis Vector DB

This notebook:
1. Loads documents from various sources (files, URLs, plain text)
2. Splits them into chunks
3. Embeds chunks with OpenAI embeddings
4. Stores vectors in Redis
5. Builds a RAG retrieval chain with Claude LLM
6. Exposes a helper that matches the `/api/chat` response schema expected by the Next.js frontend

**Requirements:** Redis running locally or via Docker  
```bash
docker run -d --name redis-rag -p 6379:6379 redis/redis-stack:latest
```

## 1. Install Dependencies

In [ ]:
%pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-anthropic \
    langchain-redis \
    redis \
    redisvl \
    pypdf \
    beautifulsoup4 \
    httpx \
    python-dotenv \
    tiktoken

## 2. Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env in the project root

# ── API Keys ──────────────────────────────────────────────────────────────────
OPENAI_API_KEY   = os.environ.get("OPENAI_API_KEY", "<your-openai-key>")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "<your-anthropic-key>")

# ── Redis ─────────────────────────────────────────────────────────────────────
REDIS_URL        = os.environ.get("REDIS_URL", "redis://localhost:6379")
REDIS_INDEX_NAME = os.environ.get("REDIS_INDEX_NAME", "rag-docs")

# ── Models ────────────────────────────────────────────────────────────────────
EMBEDDING_MODEL  = "text-embedding-3-small"   # OpenAI embedding model
LLM_MODEL        = "claude-sonnet-4-6"        # Claude model for generation

# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_SIZE       = 800
CHUNK_OVERLAP    = 100

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K            = 5    # number of chunks to retrieve per query

os.environ["OPENAI_API_KEY"]    = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print("Config loaded.")
print(f"  Redis       : {REDIS_URL}  index={REDIS_INDEX_NAME}")
print(f"  Embedding   : {EMBEDDING_MODEL}")
print(f"  LLM         : {LLM_MODEL}")
print(f"  Chunk size  : {CHUNK_SIZE} / overlap {CHUNK_OVERLAP}")

## 3. Load Documents

Choose one or more loaders below. Mix and match for your use case.

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    WebBaseLoader,
    DirectoryLoader,
)

raw_docs = []

# ── Option A: PDF files ───────────────────────────────────────────────────────
# pdf_loader = PyPDFLoader("../data/your_document.pdf")
# raw_docs += pdf_loader.load()

# ── Option B: Directory of .txt / .md files ───────────────────────────────────
# dir_loader = DirectoryLoader("../data/", glob="**/*.txt", loader_cls=TextLoader)
# raw_docs += dir_loader.load()

# ── Option C: Web URLs ────────────────────────────────────────────────────────
# web_loader = WebBaseLoader(["https://example.com/docs"])
# raw_docs += web_loader.load()

# ── Option D: Inline text (demo / quick test) ─────────────────────────────────
from langchain_core.documents import Document

raw_docs += [
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) is an AI framework that combines "
            "large language models with external knowledge retrieval. Instead of relying "
            "solely on parametric knowledge baked into model weights, RAG fetches "
            "relevant documents at inference time and conditions the LLM response on them."
        ),
        metadata={"source": "inline", "title": "What is RAG?"},
    ),
    Document(
        page_content=(
            "Redis Stack extends Redis with modules including RedisSearch and RedisJSON. "
            "RedisSearch enables full-text and vector similarity search, making Redis a "
            "fast, in-memory vector store suitable for RAG workloads at low latency."
        ),
        metadata={"source": "inline", "title": "Redis as a Vector Store"},
    ),
    Document(
        page_content=(
            "LangChain is a framework for building LLM-powered applications. It provides "
            "composable abstractions for document loading, text splitting, embedding, "
            "vector stores, and retrieval chains, enabling developers to build RAG "
            "pipelines with minimal boilerplate."
        ),
        metadata={"source": "inline", "title": "LangChain Overview"},
    ),
]

print(f"Loaded {len(raw_docs)} document(s).")
for doc in raw_docs:
    print(f"  [{doc.metadata.get('title', doc.metadata.get('source', '?'))}] {len(doc.page_content)} chars")

## 4. Split into Chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_docs)

print(f"Split into {len(chunks)} chunk(s).")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i:02d} | {len(chunk.page_content)} chars | {chunk.metadata}")

## 5. Embedding Model

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

# Smoke-test: embed a single sentence
test_vector = embeddings.embed_query("Hello, RAG!")
print(f"Embedding dimension: {len(test_vector)}")

## 6. Store Embeddings in Redis Vector DB

In [ ]:
from langchain_redis import RedisVectorStore
from redisvl.schema import IndexSchema

# Define the Redis index schema
schema = IndexSchema.from_dict({
    "index": {
        "name": REDIS_INDEX_NAME,
        "prefix": f"{REDIS_INDEX_NAME}:doc",
    },
    "fields": [
        {"name": "id",           "type": "tag"},
        {"name": "content",      "type": "text"},
        {"name": "content_vector", "type": "vector", "attrs": {
            "algorithm": "hnsw",
            "dims": len(test_vector),
            "distance_metric": "cosine",
            "datatype": "float32",
        }},
        {"name": "source",       "type": "tag"},
        {"name": "title",        "type": "text"},
    ],
})

vector_store = RedisVectorStore(
    embeddings=embeddings,
    index_schema=schema,
    redis_url=REDIS_URL,
)

print("Redis vector store initialised.")

In [ ]:
# Add documents — re-run this cell whenever your data changes.
# To avoid duplicates on re-runs, drop the index first:
#   vector_store.index.delete(drop=True)

ids = vector_store.add_documents(chunks)

print(f"Embedded and stored {len(ids)} chunk(s) in Redis index '{REDIS_INDEX_NAME}'.")
print("Sample IDs:", ids[:3])

## 7. Verify — Similarity Search

In [ ]:
query = "What is RAG and how does it work?"
results = vector_store.similarity_search_with_score(query, k=TOP_K)

print(f"Top {TOP_K} results for: '{query}'\n")
for doc, score in results:
    title = doc.metadata.get("title", doc.metadata.get("source", "?"))
    print(f"  [{score:.4f}] {title}")
    print(f"           {doc.page_content[:120]}...")
    print()

## 8. Build the RAG Retrieval Chain (Claude LLM)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ── Retriever ────────────────────────────────────────────────────────────────
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

# ── LLM ──────────────────────────────────────────────────────────────────────
llm = ChatAnthropic(
    model=LLM_MODEL,
    anthropic_api_key=ANTHROPIC_API_KEY,
    temperature=0.2,
    max_tokens=1024,
)

# ── Prompt ───────────────────────────────────────────────────────────────────
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the user's question using ONLY the "
     "information in the provided context. If the context does not contain "
     "enough information, say so — do not make things up.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[{i+1}] {doc.metadata.get('title', 'Document')}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

# ── Chain ─────────────────────────────────────────────────────────────────────
rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## 9. Test the RAG Chain

In [ ]:
question = "How does Redis help with RAG?"

# Retrieve sources separately so we can surface them
source_docs = retriever.invoke(question)
answer = rag_chain.invoke(question)

print(f"Q: {question}\n")
print(f"A: {answer}\n")
print("Sources:")
for doc in source_docs:
    print(f"  - {doc.metadata.get('title', doc.metadata.get('source', '?'))}")

## 10. Frontend-Compatible Helper

Returns the same `{ message, sources[] }` shape consumed by the Next.js `/api/chat` route.

In [ ]:
import uuid
from typing import TypedDict, Optional

class Source(TypedDict):
    id: str
    title: str
    content: str
    url: Optional[str]
    score: Optional[float]

class ChatResponse(TypedDict):
    message: str
    sources: list[Source]


def chat(query: str, conversation_history: list[dict] | None = None) -> ChatResponse:
    """
    Run a RAG query and return a response compatible with the Next.js frontend.

    Args:
        query: The user's question.
        conversation_history: Optional prior messages [{role, content}].
                              (Currently used for context display; extend
                               the prompt template to inject history.)

    Returns:
        { message: str, sources: Source[] }
    """
    # Retrieve relevant documents with scores
    source_docs_with_scores = vector_store.similarity_search_with_score(query, k=TOP_K)

    # Build context string for the LLM
    context = format_docs([doc for doc, _ in source_docs_with_scores])

    # Generate answer
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context,
        "question": query,
    })

    # Build sources list matching the frontend Source type
    sources: list[Source] = [
        Source(
            id=str(uuid.uuid4()),
            title=doc.metadata.get("title", doc.metadata.get("source", "Document")),
            content=doc.page_content[:300],
            url=doc.metadata.get("url") or doc.metadata.get("source") if doc.metadata.get("source", "").startswith("http") else None,
            score=float(score),
        )
        for doc, score in source_docs_with_scores
    ]

    return ChatResponse(message=answer, sources=sources)


# ── Quick test ────────────────────────────────────────────────────────────────
import json

response = chat("Explain LangChain and its role in RAG pipelines.")
print(json.dumps(response, indent=2))

## 11. (Optional) FastAPI Server

Drop this into `backend/main.py` to serve the RAG chain as the HTTP backend  
that the Next.js frontend calls at `RAG_BACKEND_URL/chat`.

```python
# backend/main.py
from fastapi import FastAPI
from pydantic import BaseModel
# import the chat() helper from this notebook (or refactor into a module)

app = FastAPI()

class ChatRequest(BaseModel):
    query: str
    messages: list[dict] = []

@app.post("/chat")
def chat_endpoint(req: ChatRequest):
    return chat(req.query, req.messages)
```

Run with:
```bash
uvicorn backend.main:app --reload --port 8000
```

## 12. Index Management Utilities

In [ ]:
# ── Check index info ───────────────────────────────────────────────────────────
info = vector_store.index.info()
print("Index info:")
print(f"  name          : {info.get('index_name')}")
print(f"  num_docs      : {info.get('num_docs')}")
print(f"  indexing_failures : {info.get('hash_indexing_failures')}")

In [ ]:
# ── Drop and recreate the index (run manually when needed) ────────────────────
# WARNING: this deletes all stored vectors for this index.
#
# vector_store.index.delete(drop=True)
# print("Index dropped.")